In [1]:
import pandas as pd
import requests

In [2]:
url = 'https://remoteok.com/api?tag=data'
headers = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        ' (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    )
}

In [3]:
response = requests.get(url, headers=headers)

In [7]:
if response.status_code == 200:
    data = response.json()
    raw_jobs_data = []

    for job in data[1:]:
        raw_jobs_data.append({
            'Job Title': job.get('position', 'N/A'),
            'Company': job.get('company', 'N/A'),
            'Location': job.get('location', 'Worldwide'),
            'Skills / Tech Stack': ', '.join(job.get('tags', []))
            if job.get('tags')
            else 'Not specified',
            'Post Date': job.get('date', 'N/A'),
        })
        
    df_raw = pd.DataFrame(raw_jobs_data)
    print('Məlumatlar saytdan çəkilir...')
    print(
        f'{len(df_raw)} xam məlumat toplandı.\n'
    )
    display(df_raw)
else:
    print('Sorğuda xəta baş verdi:', response.status_code)

Məlumatlar saytdan çəkilir...
100 xam məlumat toplandı.



,Job Title,Company,Location,Skills / Tech Stack,Post Date
0,Administrative Assistant,TIMBERTEK,"Ø¯Ø¨Ù, Ø¯Ø¨Ù Ø§ÙØ¥Ù Ø§Ø±Ø§Øª Ø§ÙØ¹Ø±Ø¨ÙØ©...","data entry, virtual assistant, teaching, educa...",2026-08-04T11:00:10+00:00
1,Assistant Area Sales &amp; Customer Manager,Unilever,"Navi Mumbai, Navi Mumbai, Maharashtra, India","data entry, virtual assistant, teaching, educa...",2026-08-03T15:40:30+00:00
2,Customer Support,Re Lytics Hires,"Ø¹Ø³ÙØ±, Ø¹Ø³ÙØ± Ø§ÙØ³Ø¹ÙØ¯ÙØ©","data entry, virtual assistant, teaching, educa...",2026-08-03T07:13:47+00:00
3,Entry Level Administrative Assistant,Re Lytics Hires,"Ø£Ø¨Ù Ø¸Ø¨Ù, Ø£Ø¨Ù Ø¸Ø¨Ù Ø£Ø¨Ù Ø¸Ø¨Ù Ø§Ù...","data entry, virtual assistant, teaching, educa...",2026-08-03T06:46:44+00:00
4,Data Entry &amp; Administrative Assistant,Re Lytics Hires,"Ø£Ø¨Ù Ø¸Ø¨Ù, Ø£Ø¨Ù Ø¸Ø¨Ù Ø£Ø¨Ù Ø¸Ø¨Ù Ø§Ù...","data entry, virtual assistant, teaching, educa...",2026-08-03T06:36:53+00:00
...,...,...,...,...,...
95,Designer,Cats Protection,"Chelwood Gate,","analyst, customer support, marketing, travel, ...",2026-06-30T15:34:07+00:00
96,Data Entry Specialist Assistant Administrator,Recruit Lytics Hiring,"Hong Kong, Hong Kong, Hong Kong SAR","virtual assistant, education, customer support...",2026-06-26T07:47:28+00:00
97,Nurse Advice Swing Shift 5a 10a 10p 5a CST,IntellaTriage,"United States,","hr, virtual assistant, exec, customer support,...",2026-06-24T08:09:31+00:00
98,Director of Procurement,Kardion,"Remote,","data entry, infosec, customer support, dev, ex...",2026-06-24T05:07:07+00:00


In [9]:
print('🔍 Xam Məlumatın Yoxlanılması:\n')

amp_count = df_raw['Job Title'].str.contains('&amp;').sum()
print(f" '&amp;' simvolu olan sətir sayı: {amp_count}")

sample_date = df_raw['Post Date'].iloc[0]
print(f' Nümunə tarix formatı: {sample_date}')

print('\n--- Uyğunsuz Simvolların Nümunəsi ---')
display(df_raw[['Job Title', 'Company', 'Location']].head(5))

🔍 Xam Məlumatın Yoxlanılması:

 '&amp;' simvolu olan sətir sayı: 2
 Nümunə tarix formatı: 2026-08-04T11:00:10+00:00

--- Uyğunsuz Simvolların Nümunəsi ---


,Job Title,Company,Location
0,Administrative Assistant,TIMBERTEK,"Ø¯Ø¨Ù, Ø¯Ø¨Ù Ø§ÙØ¥Ù Ø§Ø±Ø§Øª Ø§ÙØ¹Ø±Ø¨ÙØ©..."
1,Assistant Area Sales &amp; Customer Manager,Unilever,"Navi Mumbai, Navi Mumbai, Maharashtra, India"
2,Customer Support,Re Lytics Hires,"Ø¹Ø³ÙØ±, Ø¹Ø³ÙØ± Ø§ÙØ³Ø¹ÙØ¯ÙØ©"
3,Entry Level Administrative Assistant,Re Lytics Hires,"Ø£Ø¨Ù Ø¸Ø¨Ù, Ø£Ø¨Ù Ø¸Ø¨Ù Ø£Ø¨Ù Ø¸Ø¨Ù Ø§Ù..."
4,Data Entry &amp; Administrative Assistant,Re Lytics Hires,"Ø£Ø¨Ù Ø¸Ø¨Ù, Ø£Ø¨Ù Ø¸Ø¨Ù Ø£Ø¨Ù Ø¸Ø¨Ù Ø§Ù..."


In [10]:
import html
df_clean = df_raw.copy()

df_clean['Job Title'] = df_clean['Job Title'].apply(html.unescape)
df_clean['Company'] = df_clean['Company'].apply(html.unescape)

def clean_location(loc):
    if loc and not str(loc).isascii():
        try:
            return str(loc).encode('latin1').decode('utf-8')
        except Exception:
            return 'Worldwide'
    return loc if loc else 'Worldwide'

df_clean['Location'] = df_clean['Location'].apply(clean_location)

df_clean['Post Date'] = (
    df_clean['Post Date'].astype(str).str.split('T').str[0]
)

df_clean.to_csv('remote_job_market_data.csv', index=False, encoding='utf-8-sig')


In [11]:
print('Məlumatlar tam təmizləndi və remote_job_market_data.csv faylına yazıldı.\n')
display(df_clean)

Məlumatlar tam təmizləndi və remote_job_market_data.csv faylına yazıldı.



,Job Title,Company,Location,Skills / Tech Stack,Post Date
0,Administrative Assistant,TIMBERTEK,"دبي, دبي الإمارات العربية المتحدة","data entry, virtual assistant, teaching, educa...",2026-08-04
1,Assistant Area Sales & Customer Manager,Unilever,"Navi Mumbai, Navi Mumbai, Maharashtra, India","data entry, virtual assistant, teaching, educa...",2026-08-03
2,Customer Support,Re Lytics Hires,"عسير, عسير السعودية","data entry, virtual assistant, teaching, educa...",2026-08-03
3,Entry Level Administrative Assistant,Re Lytics Hires,"أبو ظبي, أبو ظبي أبو ظبي الإمارات العربية المتحدة","data entry, virtual assistant, teaching, educa...",2026-08-03
4,Data Entry & Administrative Assistant,Re Lytics Hires,"أبو ظبي, أبو ظبي أبو ظبي الإمارات العربية المتحدة","data entry, virtual assistant, teaching, educa...",2026-08-03
...,...,...,...,...,...
95,Designer,Cats Protection,"Chelwood Gate,","analyst, customer support, marketing, travel, ...",2026-06-30
96,Data Entry Specialist Assistant Administrator,Recruit Lytics Hiring,"Hong Kong, Hong Kong, Hong Kong SAR","virtual assistant, education, customer support...",2026-06-26
97,Nurse Advice Swing Shift 5a 10a 10p 5a CST,IntellaTriage,"United States,","hr, virtual assistant, exec, customer support,...",2026-06-24
98,Director of Procurement,Kardion,"Remote,","data entry, infosec, customer support, dev, ex...",2026-06-24
